# Full-wave simulation geometry, S-parameter, impedance, efficiency, ECC, diversity gain, and radiation pattern data for a four-port modified petal-like MIMO antenna from mmWave to sub-THz frequencies Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

This notebook is designed to:
- load the Croissant metadata from the provided URL,
- inspect the dataset structure using entity `@id` values,
- discover available record sets and fields dynamically,
- load record sets into pandas DataFrames when present,
- perform lightweight exploratory data analysis (EDA), and
- create example visualizations for any tabular data that can be loaded.

> **Important:** In Croissant datasets, record sets, fields, file objects, and related entities should be referenced by their `@id`. This notebook follows that rule throughout.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

We keep the Croissant schema URL in a variable so the notebook is easy to reuse for other datasets.

In [ ]:
import json
import warnings

import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.mq0y-paam/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset loaded successfully.")
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")

In [ ]:
# Convert metadata to JSON only for display or inspection.
# We avoid treating dataset.metadata itself like a dictionary.
metadata_json = metadata.to_json()

summary_fields = [
    "@id",
    "name",
    "identifier",
    "version",
    "license",
    "datePublished",
    "conformsTo",
]

metadata_summary = {k: metadata_json.get(k) for k in summary_fields if k in metadata_json}
metadata_summary

## 2. Data Overview
Review available record sets, fields, and their IDs.

This step is especially important because Croissant datasets can contain multiple record sets, each with different schemas. We inspect the metadata dynamically and always display the `@id` values so they can be used consistently in later steps.

In [ ]:
# Explore top-level metadata keys from the JSON representation
sorted(metadata_json.keys())[:50]

In [ ]:
# Extract record sets from the JSON representation.
record_sets_json = metadata_json.get("recordSet", []) or []

print(f"Number of record sets declared in metadata: {len(record_sets_json)}")

record_set_rows = []
for rs in record_sets_json:
    record_set_rows.append({
        "record_set_id": rs.get("@id"),
        "name": rs.get("name"),
        "description": rs.get("description"),
        "field_count": len(rs.get("field", []) or []),
    })

record_sets_df = pd.DataFrame(record_set_rows)
record_sets_df

If record sets are available, the next cell expands them and lists their fields, including field `@id`, name, source, and data type where possible.

In [ ]:
field_rows = []

for rs in record_sets_json:
    rs_id = rs.get("@id")
    rs_name = rs.get("name")
    for field in rs.get("field", []) or []:
        source = field.get("source")
        source_ref = None
        if isinstance(source, dict):
            source_ref = source.get("@id")
        elif isinstance(source, str):
            source_ref = source
        
        field_rows.append({
            "record_set_id": rs_id,
            "record_set_name": rs_name,
            "field_id": field.get("@id"),
            "field_name": field.get("name"),
            "description": field.get("description"),
            "data_type": field.get("dataType"),
            "source_id": source_ref,
        })

fields_df = pd.DataFrame(field_rows)
fields_df.head(20)

In addition to record sets, Croissant metadata often contains file objects and distribution entries. These can help explain where the tabular data originates.

In [ ]:
distribution_json = metadata_json.get("distribution", []) or []

distribution_rows = []
for item in distribution_json:
    distribution_rows.append({
        "distribution_id": item.get("@id"),
        "name": item.get("name"),
        "contentUrl": item.get("contentUrl"),
        "encodingFormat": item.get("encodingFormat"),
        "description": item.get("description"),
    })

distribution_df = pd.DataFrame(distribution_rows)
distribution_df

The next cell demonstrates how to inspect records from a specific record set using its `@id`.

The code is written defensively: if the dataset has no declared record sets, it will simply report that instead of failing.

In [ ]:
if record_sets_json:
    example_record_set_id = record_sets_json[0].get("@id")
    print(f"Inspecting records for record set @id: {example_record_set_id}")
    
    try:
        for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
            print(record)
            if i >= 2:
                break
    except Exception as e:
        print(f"Could not iterate over records for {example_record_set_id}: {e}")
else:
    print("No record sets are declared in the dataset metadata.")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis.

We dynamically collect all available record set `@id` values and attempt to materialize each one into a pandas DataFrame. This is useful when the exact record set structure is not known in advance.

In [ ]:
# Extract data from each record set using record set @id values
record_set_ids = [rs.get("@id") for rs in record_sets_json if rs.get("@id")]
dataframes = {}
load_errors = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} records from {record_set_id}")
    except Exception as e:
        load_errors[record_set_id] = str(e)
        print(f"Failed to load {record_set_id}: {e}")

print("\nLoaded DataFrames:", list(dataframes.keys()))
print("Load errors:", load_errors)

In [ ]:
if dataframes:
    sample_record_set_id = list(dataframes.keys())[0]
    print(f"Sample record set @id: {sample_record_set_id}")
    print("Columns:")
    print(dataframes[sample_record_set_id].columns.tolist())
    display(dataframes[sample_record_set_id].head())
else:
    print("No record sets could be loaded into DataFrames.")

For convenience, the following helper tables summarize the shape and columns of each loaded record set.

In [ ]:
dataframe_summary_rows = []

for record_set_id, df in dataframes.items():
    dataframe_summary_rows.append({
        "record_set_id": record_set_id,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "column_names": list(df.columns),
    })

dataframe_summary_df = pd.DataFrame(dataframe_summary_rows)
dataframe_summary_df

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Because the exact structure of the dataset may differ across record sets, this section automatically identifies a suitable numeric field if one exists. It then:
- selects a numeric column,
- filters records above a threshold,
- computes a normalized version of the field, and
- optionally groups by a categorical field.

This keeps the notebook robust while still following the template's analytical workflow.

In [ ]:
def find_numeric_and_group_fields(df: pd.DataFrame):
    numeric_candidates = [
        c for c in df.columns
        if pd.api.types.is_numeric_dtype(df[c]) and df[c].notna().sum() > 0
    ]
    
    group_candidates = [
        c for c in df.columns
        if (df[c].dtype == 'object' or pd.api.types.is_categorical_dtype(df[c]))
        and df[c].nunique(dropna=True) > 1
        and df[c].nunique(dropna=True) <= min(20, len(df))
    ]
    
    numeric_field = numeric_candidates[0] if numeric_candidates else None
    group_field = group_candidates[0] if group_candidates else None
    return numeric_field, group_field


eda_results = {}

for record_set_id, df in dataframes.items():
    print(f"\n--- EDA for record set @id: {record_set_id} ---")
    
    if df.empty:
        print("DataFrame is empty; skipping.")
        continue
    
    working_df = df.copy()
    numeric_field, group_field = find_numeric_and_group_fields(working_df)
    
    if numeric_field is None:
        print("No numeric field found for this record set; skipping numeric EDA.")
        continue
    
    numeric_series = working_df[numeric_field].dropna()
    if numeric_series.empty:
        print(f"Numeric field {numeric_field} contains no non-null values; skipping.")
        continue
    
    threshold = numeric_series.median()
    filtered_df = working_df[working_df[numeric_field] > threshold].copy()
    
    print(f"Selected numeric field: {numeric_field}")
    print(f"Threshold (median): {threshold}")
    print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)}")
    display(filtered_df.head())
    
    std_value = filtered_df[numeric_field].std()
    if pd.notna(std_value) and std_value != 0:
        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean())
            / filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, normalized_col]].head())
    else:
        normalized_col = None
        print(f"Standard deviation is zero or undefined for {numeric_field}; normalization skipped.")
    
    grouped_df = None
    if group_field and group_field in filtered_df.columns:
        numeric_cols = filtered_df.select_dtypes(include='number').columns.tolist()
        if numeric_cols:
            grouped_df = filtered_df.groupby(group_field)[numeric_cols].mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
    else:
        print("No suitable grouping field found for this record set.")
    
    eda_results[record_set_id] = {
        "numeric_field": numeric_field,
        "group_field": group_field,
        "threshold": threshold,
        "filtered_df": filtered_df,
        "normalized_col": normalized_col,
        "grouped_df": grouped_df,
    }

As an additional preprocessing example, the next cell reports missingness and attempts a simple outlier screen using the interquartile range (IQR) rule for numeric columns.

In [ ]:
for record_set_id, df in dataframes.items():
    print(f"\n--- Data quality summary for {record_set_id} ---")
    
    if df.empty:
        print("Empty DataFrame.")
        continue
    
    missing_summary = df.isna().sum().sort_values(ascending=False)
    display(missing_summary.to_frame("missing_values").head(10))
    
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    if not numeric_cols:
        print("No numeric columns available for outlier screening.")
        continue
    
    outlier_report = []
    for col in numeric_cols[:10]:
        series = df[col].dropna()
        if len(series) < 4:
            continue
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            outlier_count = 0
        else:
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            outlier_count = ((series < lower) | (series > upper)).sum()
        outlier_report.append({
            "column": col,
            "outlier_count": int(outlier_count),
            "non_null_count": int(series.shape[0]),
        })
    
    if outlier_report:
        display(pd.DataFrame(outlier_report).sort_values("outlier_count", ascending=False))
    else:
        print("Not enough numeric data for outlier screening.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The following cells generate plots when suitable numeric and grouping fields are available. They are written to adapt to whichever record sets were successfully loaded.

In [ ]:
sns.set_theme(style="whitegrid")

for record_set_id, result in eda_results.items():
    filtered_df = result["filtered_df"]
    numeric_field = result["numeric_field"]
    group_field = result["group_field"]
    normalized_col = result["normalized_col"]
    
    if filtered_df is None or filtered_df.empty or numeric_field is None:
        continue
    
    print(f"\nVisualizations for record set @id: {record_set_id}")
    
    fig, axes = plt.subplots(1, 2 if normalized_col else 1, figsize=(14, 4))
    if not isinstance(axes, (list, tuple)):
        try:
            axes = axes.flatten()
        except Exception:
            axes = [axes]
    
    sns.histplot(filtered_df[numeric_field].dropna(), kde=True, ax=axes[0], color="steelblue")
    axes[0].set_title(f"Distribution of {numeric_field}")
    axes[0].set_xlabel(numeric_field)
    
    if normalized_col:
        sns.histplot(filtered_df[normalized_col].dropna(), kde=True, ax=axes[1], color="darkorange")
        axes[1].set_title(f"Distribution of {normalized_col}")
        axes[1].set_xlabel(normalized_col)
    
    plt.tight_layout()
    plt.show()
    
    if group_field and group_field in filtered_df.columns:
        plt.figure(figsize=(10, 4))
        plot_df = filtered_df[[group_field, numeric_field]].dropna()
        if not plot_df.empty:
            sns.boxplot(data=plot_df, x=group_field, y=numeric_field)
            plt.title(f"{numeric_field} by {group_field}")
            plt.xticks(rotation=45, ha="right")
            plt.tight_layout()
            plt.show()

If no record sets were available or loadable, the dataset can still be explored through its metadata. The next cell provides a simple metadata-driven visualization using keyword frequencies.

In [ ]:
keywords = metadata_json.get("keywords", []) or []

if keywords:
    keyword_df = pd.DataFrame({"keyword": keywords})
    plt.figure(figsize=(10, max(4, len(keyword_df) * 0.25)))
    sns.countplot(data=keyword_df, y="keyword", order=keyword_df["keyword"].value_counts().index, color="teal")
    plt.title("Dataset keywords")
    plt.xlabel("Count")
    plt.ylabel("Keyword")
    plt.tight_layout()
    plt.show()
else:
    print("No keywords found in metadata.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we used `mlcroissant` to load a Croissant dataset from its schema URL and inspect its metadata structure.

Key takeaways:
- The dataset metadata was loaded directly as a `mlcroissant` metadata object.
- We converted metadata to JSON only when needed for display and structured inspection.
- All dataset entities were referenced using their `@id` values whenever possible.
- The notebook dynamically searched for record sets and fields, then attempted to load records into pandas DataFrames.
- When tabular records were available, we demonstrated filtering, normalization, grouping, data quality checks, and visualizations.
- Even if a dataset exposes limited or no record sets, Croissant metadata still provides a rich basis for exploration.

For this specific FAIR^2 dataset, the metadata describes a simulation-based antenna dataset focused on CST Studio Suite outputs, design evolution, parametric studies, S-parameters, impedance, radiation efficiency, ECC, diversity gain, and radiation patterns across mmWave to sub-THz frequencies.

Possible next steps:
- inspect file-level distribution entries in more detail,
- identify any future record sets added to the Croissant schema,
- map field `@id` values to domain-specific antenna variables,
- build surrogate models for antenna response prediction, and
- compare simulation outputs across antenna geometries and parameter sweeps.